# Soup Kernel Challenge: T4 benchmark — adilzhanw

Kaggle settings (right panel): **Accelerator: GPU T4**, **Internet: on** (needs a phone-verified Kaggle account).
This notebook clones our repo, installs the pinned deps, runs the base vs. candidate benchmark
(`harness/run_task.py`), then the 10 rule checks (`harness/check_rules.py`). About 12-18 min.

For the submission: set this notebook to **public**, pin `COMMIT` below to the final commit,
**Save Version**, and put the `?scriptVersionId=` link in `SUBMISSION.md` and the README.
Do not run other GPU work here while the timed runs go — the T4 must stay idle.

In [ ]:
REPO_URL = "https://github.com/adilzhanw/soup-kernel-solution"  # our repository
COMMIT = "93da5af7729dbcafdfddb66bb0b470d6b8905cd3"  # the commit we submit

In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print("GPU:", gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 (right panel)"
# optional HF token via Kaggle secrets (avoids HF rate limits)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
# pinned deps that the task expects (torch/triton are preinstalled on the T4 image)
!python -m pip install -q "transformers==4.46.3" "peft==0.12.0" "trl==0.14.0" \
    "bitsandbytes==0.43.3" "accelerate==1.14.0" "datasets==2.21.0" \
    "huggingface_hub==0.24.7" "sentencepiece" "protobuf" "soups-cli==0.77.0"

In [ ]:
import subprocess, os
os.chdir("/kaggle/working")
r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-2000:])
os.chdir("/kaggle/working/repo")
if COMMIT:
    # full history is not available with --depth 1; re-fetch to check out the pinned commit
    subprocess.run(["git", "fetch", "--unshallow"], capture_output=True)
    c = subprocess.run(["git", "checkout", COMMIT], capture_output=True, text=True)
    print(c.stdout[-500:]); print(c.stderr[-500:])
!python task/prepare_data.py
!python -m pytest solution -q

In [ ]:
# base then candidate in one session -> runs/result.json (wall time, peak memory, eval loss, speedup, memory ratio)
!python harness/run_task.py

In [ ]:
# the 10 rule checks -> runs/checks.json (exit 0 = all passed)
!python harness/check_rules.py

In [ ]:
# save the result for the submission
!cat runs/result.json | python -m json.tool | head -60
!echo '---- checks ----'
!cat runs/checks.json | python -m json.tool | head -60

Quick-iteration variant (only the candidate, ~3 min; checks 1 and 5-8 need the base so they fail here):
`python harness/run_task.py --only solution --out runs_try`
Add `--profile` to get a per-kernel GPU-time table in `runs/solution_profile/profile.json`
(used to show our Triton kernels' share of the step for the video).
Per-run logs: `runs/<run>/train.log`.